Build the Neural Network

In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

Get Device for Training

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


Define the Class

In [9]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28,512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [12]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [13]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([4])


Model Layers

In [14]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [15]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [16]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [17]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-1.0786e-01,  5.3432e-01,  7.7875e-01,  5.5405e-01, -7.3287e-02,
          2.0199e-01, -2.5074e-01, -6.9084e-02, -2.7100e-01,  3.5262e-01,
         -1.4162e-01, -3.1004e-01, -1.4534e-01, -1.1552e-01,  1.6930e-01,
         -3.3231e-02,  3.4168e-02,  5.1494e-02, -1.1119e-01,  5.0448e-01],
        [-3.7444e-01,  6.2981e-01,  7.2122e-01,  5.1595e-01, -1.6747e-01,
          3.5383e-01, -7.8562e-03, -2.7550e-01, -6.0108e-01,  5.2765e-01,
         -4.4313e-02, -1.7214e-01,  1.3790e-01, -7.5910e-02,  2.7293e-01,
         -4.8138e-01, -1.4878e-01,  3.4184e-01, -2.2548e-01,  2.6443e-01],
        [-1.7799e-01,  3.9339e-01,  6.1443e-01,  6.4607e-01, -2.8889e-01,
          4.1963e-01, -1.7956e-06, -2.0511e-01, -2.4812e-01,  5.9633e-01,
         -1.6784e-01, -3.4885e-01,  1.4710e-01, -2.3319e-01,  3.6854e-01,
         -3.8850e-01, -4.2810e-01,  3.3137e-01, -8.0248e-02,  2.3445e-01]],
       grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.5343, 0.7787, 0.5541, 0.0000,

In [18]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

In [19]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

Model Parameters

In [20]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0099, -0.0169, -0.0011,  ...,  0.0034,  0.0135, -0.0147],
        [-0.0298, -0.0102,  0.0144,  ..., -0.0181,  0.0256, -0.0187]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([ 0.0156, -0.0355], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0157, -0.0144,  0.0048,  ...,  0.0073, -0.0360, -0.0080],
        [-0.0010,  0.0126,  0.0345,  ..., -0.0167, -0.0227, -0.0370]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | 